# FoodWise — Data Preprocessing Pipeline
This notebook walks through the data cleaning, standardization, and feature engineering steps implemented in `src/preprocessing.py`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Import preprocessing pipeline
sys.path.append("..")
from src.preprocessing import load_raw_data, clean_dataset

raw_df = load_raw_data()
print("Raw Dataset Loaded. Shape:", raw_df.shape)

Raw Dataset Loaded. Shape: (44891, 7)


## Step-by-Step Data Cleaning Execution
1. Remove exact duplicate rows.
2. Filter non-restaurant entities (e.g. pharmacy listings) and permanently closed entries.
3. Parse numeric prices into `price_numeric` and assign `price_tier`.
4. Parse ratings into `rating_numeric` and record `rating_status` ('Rated', 'New', 'Unrated').
5. Extract delivery duration in minutes and operational status.
6. Build normalized `content_soup` for recommendation modeling.

In [2]:
cleaned_df = clean_dataset(raw_df)
print("Cleaned Dataset Shape:", cleaned_df.shape)
print("\nColumns in Cleaned Dataset:")
print(cleaned_df.columns.tolist())
cleaned_df[["Restaurant Name", "city", "rating_numeric", "price_numeric", "price_tier", "delivery_time_mins", "safety_label", "is_pure_veg"]].head()

Cleaned Dataset Shape: (44708, 19)

Columns in Cleaned Dataset:
['Restaurant Name', 'Rating', 'Cuisine', 'Average Price', 'Average Delivery Time', 'Safety Measure', 'Location', 'city', 'rating_numeric', 'rating_status', 'price_numeric', 'price_tier', 'delivery_time_mins', 'delivery_status', 'safety_label', 'cuisine_list', 'cuisine_clean', 'is_pure_veg', 'content_soup']


,Restaurant Name,city,rating_numeric,price_numeric,price_tier,delivery_time_mins,safety_label,is_pure_veg
0,Campus Bakers,Agra,4.3,50.0,Budget,36.0,Restaurant partner follows WHO protocol,False
1,Mama Chicken Mama Franky House,Agra,4.0,50.0,Budget,22.0,Follows all Max Safety measures to ensure your...,False
2,GMB - Gopika Sweets & Restaurant,Agra,4.2,50.0,Budget,27.0,Follows all Max Safety measures to ensure your...,False
3,Shree Bankey Bihari Misthan Bhandar,Agra,4.2,50.0,Budget,28.0,Restaurant partner follows WHO protocol,False
4,Burger King,Agra,4.2,50.0,Budget,26.0,Follows all Max Safety measures to ensure your...,False


## Validation of Cleaned Attributes

In [3]:
print("Rating Status Distribution:")
print(cleaned_df["rating_status"].value_counts())

print("\nPrice Tier Distribution:")
print(cleaned_df["price_tier"].value_counts())

print("\nDelivery Status Distribution:")
print(cleaned_df["delivery_status"].value_counts())

Rating Status Distribution:
rating_status
Rated            37758
Unrated / New     6950
Name: count, dtype: int64

Price Tier Distribution:
price_tier
Budget       26939
Mid-range    14902
Premium       2858
Unknown          9
Name: count, dtype: int64

Delivery Status Distribution:
delivery_status
Delivery time listed    40189
Unknown                  3687
Not Accepting Orders      763
Temporarily Closed         69
Name: count, dtype: int64


## Feature Engineering: Content Soup Preview
The `content_soup` combines cuisine tags (repeated twice for extra weight), city, price tier and a name-based vegetarian hint.

In [4]:
for idx in range(3):
    row = cleaned_df.iloc[idx]
    print(f"Restaurant: {row['Restaurant Name']}")
    print(f"Content Soup: {row['content_soup']}\n")

Restaurant: Campus Bakers
Content Soup: bakery fast food pizza sandwich burger bakery fast food pizza sandwich burger agra budget

Restaurant: Mama Chicken Mama Franky House
Content Soup: north indian mughlai rolls burger momos north indian mughlai rolls burger momos agra budget

Restaurant: GMB - Gopika Sweets & Restaurant
Content Soup: north indian south indian chinese fast food street food north indian south indian chinese fast food street food agra budget



## Name-based veg hint
`is_pure_veg` is only a hint from the restaurant name. Names that also say *Non Veg* are excluded.

In [5]:
veg = cleaned_df[cleaned_df['is_pure_veg']]
print('Listings flagged pure veg:', len(veg))
print('Flagged veg but name says non-veg:', veg['Restaurant Name'].str.contains(r'non[\s\-]*veg', case=False).sum())
veg['Restaurant Name'].sample(5, random_state=1).tolist()

Listings flagged pure veg: 852
Flagged veg but name says non-veg: 0


['Shobhaa Only Veg',
 'The Dessert Heaven - Pure Veg',
 'Sheetal Pure Veg Restaurant',
 'Veg Daawat By Behrouz',
 'Jain Samose Wala']